<div style="background:linear-gradient(135deg,#0D1B2A 0%,#1565C0 100%);padding:36px 40px;border-radius:12px;margin-bottom:8px;">
  <div style="display:flex;align-items:center;gap:16px;">
    <span style="font-size:48px;">🛰️</span>
    <div>
      <p style="color:#ADE8F4;margin:0;font-size:13px;letter-spacing:3px;text-transform:uppercase;font-family:monospace;">Computer Vision for Remote Sensing · Bangladesh</p>
      <h1 style="color:#FFFFFF;margin:4px 0 0 0;font-size:28px;font-family:monospace;">Phase 1 · Raster CV Foundations</h1>
    </div>
  </div>
  <hr style="border-color:#1E88E5;margin:20px 0 16px 0;"/>
  <div style="display:flex;gap:32px;flex-wrap:wrap;">
    <span style="color:#90CAF9;font-family:monospace;font-size:13px;">📘 Lesson <b style="color:#FFF">L7 / 7</b></span>
    <span style="color:#90CAF9;font-family:monospace;font-size:13px;">⏱ Est. Time <b style="color:#FFF">2 hrs</b></span>
    <span style="color:#90CAF9;font-family:monospace;font-size:13px;">📍 Study Area <b style="color:#FFF">Sitakunda patches</b></span>
    <span style="color:#90CAF9;font-family:monospace;font-size:13px;">📅 Date <b style="color:#FFF">2026-10-06</b></span>
  </div>
</div>
<div style="background:#1565C022;border-left:5px solid #1565C0;padding:18px 24px;border-radius:0 8px 8px 0;">
  <h2 style="color:#1565C0;margin:0 0 8px 0;font-size:18px;">📖 L7 — Visualization & QC</h2>
  <p style="margin:0;color:#1A2A3A;font-size:14px;">Learn to inspect a multi-band GeoTIFF before training: profile its metadata and band statistics, build RGB composites, and check NoData and patch quality.</p>
</div>

Before any model sees your data, **you** should see it first.
Visualization and QC catch problems that silently destroy training:
clipped bands, NoData floods, outlier pixels, and patch-level artifacts.
This lesson builds a complete pre-training inspection pipeline
for any multi-band GeoTIFF — run it once per dataset.

---
## ⚙️ Setup & Imports

## 🎯 Objectives

By the end of this lesson you will be able to:

- ✅ Profile a GeoTIFF: shape, CRS, bands, NoData coverage
- ✅ Plot per-band histograms and spot outlier spikes
- ✅ Build true-colour and false-colour RGB composites
- ✅ Detect and visualise NoData / cloud-masked regions
- ✅ Generate a patch QC grid to inspect training samples visually
- ✅ Compare raw vs normalised distributions in one figure

---
## 1 · Raster Profile & Band Statistics

> Read the raster's metadata (CRS, resolution, bounds, NoData value) and compute per-band min, max, mean, std and the percentage of NoData pixels. This is the first check to run on any new file.

## 🔑 Key Functions

| Function | Purpose |
|---|---|
| `src.meta` | Full raster metadata dict |
| `np.nanpercentile(arr, [2, 98])` | Robust min/max ignoring NoData |
| `ax.hist(arr.ravel(), bins=256)` | Pixel value distribution |
| `np.stack([r, g, b], axis=-1)` | Assemble RGB composite |
| `np.clip(arr, lo, hi)` | Percentile stretch for display |
| `plt.imshow(rgb)` | Render composite |
| `np.isnan(arr).any(axis=0)` | Per-pixel NoData mask across bands |

---
## 🏋️ Exercise

<div style="background:#FFF9C4;border-left:4px solid #F9A825;padding:14px 20px;border-radius:0 8px 8px 0;">
<b>Task:</b> Run the cells below to locate the test raster (local file, or a downloaded fallback), then read its profile in the next exercise cell.
</div>

In [5]:
import urllib.request, tempfile, os
import pathlib

# ── These repeat the Setup cell — safe to run standalone ─────────────────────
DATA_DIR = pathlib.Path('../../data/raw/')
FIG_DIR  = pathlib.Path('../../outputs/figures/phase1/')
FIG_DIR.mkdir(parents=True, exist_ok=True)

TIF_PATH = DATA_DIR / 'sitakunda_sentinel2.tif'

# ── Fallback to rasterio test tile if local file is missing ──────────────────
if not TIF_PATH.exists():
    print("Local file not found — downloading rasterio test tile as fallback.")
    _url = "https://raw.githubusercontent.com/rasterio/rasterio/master/tests/data/RGB.byte.tif"
    _tmp = tempfile.NamedTemporaryFile(suffix=".tif", delete=False)
    urllib.request.urlretrieve(_url, _tmp.name)
    TIF_PATH = pathlib.Path(_tmp.name)
    print(f"Using fallback: {TIF_PATH}")
else:
    print(f"Using local file: {TIF_PATH}")

print(f"TIF_PATH ready: {TIF_PATH}")

Local file not found — downloading rasterio test tile as fallback.
Using fallback: C:\Users\reza\AppData\Local\Temp\tmpxvummqim.tif
TIF_PATH ready: C:\Users\reza\AppData\Local\Temp\tmpxvummqim.tif


In [6]:
# Optional — save a persistent copy so the fallback only downloads once
import shutil
_saved = DATA_DIR / "rasterio_test_rgb.tif"
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not _saved.exists():
    shutil.copy(TIF_PATH, _saved)
    print(f"Saved permanent copy → {_saved}")
TIF_PATH = _saved

Saved permanent copy → ..\..\data\raw\rasterio_test_rgb.tif


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio

with rasterio.open(TIF_PATH) as src:
    arr     = src.read(masked=True).astype(np.float32)
    meta    = src.meta
    bounds  = src.bounds
    res     = src.res
    n_bands = src.count
    nodata  = src.nodata

img = arr.filled(np.nan)

print("═" * 54)
print("  RASTER PROFILE")
print("═" * 54)
print(f"  Bands × Rows × Cols : {img.shape}")
print(f"  Dtype               : {meta['dtype']}")
print(f"  CRS                 : {meta['crs']}")
print(f"  Pixel resolution    : {res[0]:.4f} × {res[1]:.4f}")
print(f"  NoData value        : {nodata}")
print(f"  Bounding box        : {bounds}")
print("─" * 54)
for b in range(n_bands):
    valid   = img[b][~np.isnan(img[b])]
    pct_nan = np.isnan(img[b]).mean() * 100
    print(f"  Band {b+1}  |  "
          f"min={valid.min():8.2f}  max={valid.max():8.2f}  "
          f"mean={valid.mean():8.2f}  std={valid.std():7.2f}  "
          f"NoData={pct_nan:.1f}%")
print("═" * 54)

---
## 2 · Per-Band Histograms

> Look for spikes at the extremes (0 or 255 for uint8), clipped tails, and bands whose distributions barely overlap.

In [ ]:
import matplotlib.pyplot as plt
BAND_NAMES = [f"Band {b+1}" for b in range(n_bands)]
colors = ["#e05c5c", "#5cb85c", "#5c9be0"]

fig, axes = plt.subplots(1, n_bands, figsize=(5 * n_bands, 4),
                         facecolor="#0f1117")
fig.suptitle("Per-Band Pixel Value Histograms",
             color="white", fontsize=13, y=1.02)

for b, ax in enumerate(axes):
    valid       = img[b][~np.isnan(img[b])].ravel()
    p2, p98     = np.nanpercentile(img[b], [2, 98])

    ax.set_facecolor("#1a1d27")
    ax.hist(valid, bins=200, color=colors[b % len(colors)],
            alpha=0.85, edgecolor="none")
    ax.axvline(p2,  color="white",  lw=1.2, ls="--", label=f"p2={p2:.0f}")
    ax.axvline(p98, color="yellow", lw=1.2, ls="--", label=f"p98={p98:.0f}")
    ax.set_title(BAND_NAMES[b], color="white", fontsize=11)
    ax.set_xlabel("Pixel value", color="grey", fontsize=9)
    ax.set_ylabel("Count",       color="grey", fontsize=9)
    ax.tick_params(colors="grey")
    for spine in ax.spines.values():
        spine.set_edgecolor("#333")
    ax.legend(fontsize=8, labelcolor="white", framealpha=0.2)

plt.tight_layout()
out = FIG_DIR / "L7_band_histograms.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print(f"Saved → {out}")

---
## 3 · RGB Composite

> The percentile stretch (p2–p98) maps each band to 0–1 for display. Here band 1 is red, band 2 green, band 3 blue.

In [ ]:
def percentile_stretch(band, lo=2, hi=98):
    p_lo = np.nanpercentile(band, lo)
    p_hi = np.nanpercentile(band, hi)
    return np.clip((band - p_lo) / (p_hi - p_lo + 1e-8), 0, 1)

def make_rgb(img, r_idx=0, g_idx=1, b_idx=2):
    r   = percentile_stretch(img[r_idx])
    g   = percentile_stretch(img[g_idx])
    b   = percentile_stretch(img[b_idx])
    rgb = np.stack([r, g, b], axis=-1)
    rgb[np.isnan(rgb)] = 0
    return rgb

rgb_true = make_rgb(img, r_idx=0, g_idx=1, b_idx=2)

fig, ax = plt.subplots(1, 1, figsize=(7, 7), facecolor="#0f1117")
ax.imshow(rgb_true)
ax.set_title("True Colour — Percentile Stretch (p2–p98)",
             color="white", fontsize=11)
ax.axis("off")
plt.tight_layout()
out = FIG_DIR / "L7_rgb_composite.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print(f"Saved → {out}")

---
## 4 · NoData & Coverage Map

> Flooded or cloud-masked areas appear as NoData. Check coverage before sampling patches, or empty pixels will dominate them.

In [ ]:
nodata_mask  = np.isnan(img).any(axis=0)
coverage_pct = (~nodata_mask).mean() * 100

print(f"Valid coverage : {coverage_pct:.1f}%")
print(f"NoData pixels  : {nodata_mask.sum():,} / {nodata_mask.size:,}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor="#0f1117")

rgb_annotated = rgb_true.copy()
rgb_annotated[nodata_mask] = [1, 0.2, 0.2]

axes[0].imshow(rgb_annotated)
axes[0].set_title(f"NoData highlighted red\nCoverage: {coverage_pct:.1f}%",
                  color="white", fontsize=11)
axes[0].axis("off")

axes[1].imshow(~nodata_mask, cmap="Greens", vmin=0, vmax=1)
axes[1].set_title("Coverage mask\n■ green = valid   □ black = NoData",
                  color="white", fontsize=11)
axes[1].axis("off")

fig.suptitle("NoData / Coverage Analysis", color="white", fontsize=13)
plt.tight_layout()
out = FIG_DIR / "L7_nodata_coverage.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print(f"Saved → {out}")

---
## 5 · Patch QC Grid

> Bad training samples are mostly NoData, uniform in colour, or cut off at the scene edge. Check the grid for these before training.

In [ ]:
def extract_random_patches(img, patch_size=64, n_patches=16, seed=42):
    rng     = np.random.default_rng(seed)
    C, H, W = img.shape
    patches, coords = [], []
    attempts = 0
    while len(patches) < n_patches and attempts < n_patches * 20:
        r = rng.integers(0, H - patch_size)
        c = rng.integers(0, W - patch_size)
        patch = img[:, r:r+patch_size, c:c+patch_size]
        if np.isnan(patch).mean() < 0.5:
            patches.append(patch)
            coords.append((r, c))
        attempts += 1
    print(f"Extracted {len(patches)} valid patches  "
          f"(rejected {attempts - len(patches)} high-NoData patches)")
    return patches, coords

PATCH_SIZE = 64
patches, coords = extract_random_patches(img, patch_size=PATCH_SIZE, n_patches=16)

n_cols = 4
n_rows = len(patches) // n_cols + int(len(patches) % n_cols > 0)

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(n_cols * 3, n_rows * 3),
                          facecolor="#0f1117")

for i, ax in enumerate(axes.ravel()):
    if i < len(patches):
        rgb     = make_rgb(patches[i], 0, 1, 2)
        nan_pct = np.isnan(patches[i]).mean() * 100
        ax.imshow(rgb)
        ax.set_title(f"r={coords[i][0]}, c={coords[i][1]}\n"
                     f"NoData={nan_pct:.0f}%",
                     color="white", fontsize=7, pad=3)
    ax.axis("off")

fig.suptitle(f"Patch QC Grid — {PATCH_SIZE}×{PATCH_SIZE} px  |  {len(patches)} samples",
             color="white", fontsize=12)
plt.tight_layout()
out = FIG_DIR / "L7_patch_qc_grid.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print(f"Saved → {out}")

---
## 6 · Raw vs Normalised

> Z-score centres each band at 0 with unit variance. Values beyond ±3σ show up as the tails in the right-hand panel.

In [ ]:
def zscore_normalize(arr, means=None, stds=None):
    if means is None:
        means = [float(np.nanmean(arr[b])) for b in range(arr.shape[0])]
    if stds is None:
        stds  = [float(np.nanstd(arr[b]))  for b in range(arr.shape[0])]
    normed = np.empty_like(arr)
    for b in range(arr.shape[0]):
        normed[b] = (arr[b] - means[b]) / (stds[b] + 1e-8)
    return normed, means, stds

img_zs, _, _ = zscore_normalize(img)

fig, axes = plt.subplots(n_bands, 2,
                          figsize=(12, 3.5 * n_bands),
                          facecolor="#0f1117")
if n_bands == 1:
    axes = np.expand_dims(axes, 0)

for b in range(n_bands):
    raw_valid = img[b][~np.isnan(img[b])].ravel()
    zs_valid  = img_zs[b][~np.isnan(img_zs[b])].ravel()
    outlier_pct = (np.abs(zs_valid) > 3).mean() * 100

    for ax, data, color, title in [
        (axes[b, 0], raw_valid, "#5c9be0",
         f"Band {b+1} — Raw\nμ={np.nanmean(img[b]):.1f}  σ={np.nanstd(img[b]):.1f}"),
        (axes[b, 1], zs_valid,  "#e05c5c",
         f"Band {b+1} — Z-Score\nμ={np.nanmean(img_zs[b]):.4f}  "
         f"σ={np.nanstd(img_zs[b]):.4f}  outliers>{3}σ: {outlier_pct:.1f}%"),
    ]:
        ax.set_facecolor("#1a1d27")
        ax.hist(data, bins=200, color=color, alpha=0.85, edgecolor="none")
        if color == "#e05c5c":
            ax.axvline(-3, color="yellow", lw=1, ls="--", label="±3σ")
            ax.axvline( 3, color="yellow", lw=1, ls="--")
            ax.legend(fontsize=8, labelcolor="white", framealpha=0.2)
        ax.set_title(title, color="white", fontsize=10)
        ax.tick_params(colors="grey")
        for sp in ax.spines.values():
            sp.set_edgecolor("#333")

fig.suptitle("Raw vs Z-Score Normalised Distributions",
             color="white", fontsize=13, y=1.01)
plt.tight_layout()
out = FIG_DIR / "L7_raw_vs_normalised.png"
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print(f"Saved → {out}")

---
## 📋 Phase 1 Complete

| Lesson | Topic | Status |
|---|---|---|
| L1 | Image as Tensor | ✅ |
| L2 | Spatial Profile | ✅ |
| L3 | Spectral Indices | ✅ |
| L4 | Reprojection & Resampling | ✅ |
| L5 | Patch Tiling | ✅ |
| L6 | Normalization for Deep Learning | ✅ |
| L7 | Visualization & QC | ✅ |

**→ Phase 2: Classical ML** — GLCM texture features, Random Forest,
SVM, and your first end-to-end LULC map saved as a GeoTIFF.

---
## 📝 My Notes

> Filled from this lesson's run. Edit to reflect your own observations.

- Observation 1: The test raster is 3-band uint8 in EPSG:32618 (UTM 18N) with ~300 m pixels, so resolution is in metres.
- Observation 2: 32.6% of pixels are NoData (value 0), leaving 67.3% valid coverage. The patch grid rejected 2 candidate patches for having more than 50% NoData.
- Anything unexpected: The local Sitakunda file was missing, so the exercise fell back to the rasterio test tile.

---
<div style="background:linear-gradient(135deg,#0D1B2A 0%,#1565C0 100%);padding:24px 32px;border-radius:10px;">
  <h2 style="color:#FFFFFF;margin:0 0 12px 0;font-size:18px;">✅ Key Takeaways — L7</h2>
  <ul style="color:#C8E6C9;font-size:14px;line-height:2.2;margin:0;">
    <li>Always profile a raster first: CRS, resolution, bounds and NoData tell you what the numbers mean.</li>
    <li>Compute statistics with NoData masked out, or NaN and fill values will skew every band's mean and std.</li>
    <li>Watch cell execution order: a cell that uses rasterio must import it itself or run after the import cell.</li>
  </ul>
</div>

---
<div style="display:flex;justify-content:space-between;padding:10px 0;font-size:13px;color:#9E9E9E;">
  <span>← Previous: <a href="L6_normalization.ipynb">L6</a></span>
  <b style="color:#1565C0;">Phase 1 · L7 of 7</b>
  <span>Next Phase →</span>
</div>